# 3.2 Kernel 测试指南

本指南以 FastGelu 算子为例介绍如何使用 **TTK（ops Test Tool Kit）** 工具进行 Kernel 级别测试。

> **TTK 工具地址**：https://gitcode.com/cann/ops-test-kit

1. **环境准备**：配置 CANN 开发环境和 TTK 工具
2. **用例编写**：创建 CSV 测试用例文件
3. **Golden 插件**：为 PyTorch 无内置算子创建 Golden 插件
4. **精度测试**：验证算子计算结果正确性
5. **性能测试**：采集 Profiling 性能数据

---

# 1. 环境准备

在进行 Kernel 测试前，需要完成以下环境配置工作：

---


## 1.1 基本环境配置

- **Python 版本**：建议 3.8+，安装 PyTorch（Golden 计算使用）
- **CANN 安装**：完成 CANN 包安装，确保环境变量已 source
- **自定义算子**：如测试自定义算子，请先完成编译部署

In [ ]:
# 初始化 CANN 环境
import os, subprocess

!mkdir -p TestCases

env = subprocess.check_output(
    "bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", 
    shell=True, text=True
)
for line in env.splitlines():
    if "=" in line:
        os.environ.__setitem__(*line.split("=", 1))

---

## 1.2 TTK 工具安装

TTK 是 CANN 提供的全链路、自动化、批量化算子测试框架，支持 Kernel、ACLNN、E2E 等多种测试模式。

In [ ]:
# 按照 Kernel算子测试指南.md 安装 TTK

# 检查 ops-test-kit 是否已存在
import os
if not os.path.exists('ops-test-kit'):
    !git clone https://gitcode.com/cann/ops-test-kit.git

# 安装依赖（Jupyter 中 !cd 不持久，必须用 && 连接）
!cd ops-test-kit && pip install -r requirements.txt --quiet

# 安装额外依赖（Golden 计算需要）
!pip install ml_dtypes --quiet

---
# 2. 测试用例编写

TTK 使用 CSV 文件定义测试用例，支持批量测试多个场景。

---


## 2.1 CSV 文件格式

CSV 文件的关键字段说明：

| 字段名 | 说明 | 示例 |
|--------|------|------|
| testcase_name | 测试用例名称 | fast_gelu_01 |
| op_name | 算子名称 | fast_gelu |
| input_shapes | 输入 shape 列表 | ((128, 1024),) |
| input_dtypes | 输入数据类型列表 | ('float32',) |
| input_formats | 输入内存格式列表 | ('ND',) |
| output_shapes | 输出 shape 列表 | ((128, 1024),) |
| input_data_ranges | 输入数据范围 | ((-1, 1),) |
| precision_tolerances | 精度容差 | ((0.001, 0.001),) |
| attributes | 算子属性（可选） | {} |

## 2.2 创建 FastGelu 算子测试用例

以下示例创建包含多种测试场景的 CSV 文件：

In [ ]:
# 创建 FastGelu 算子测试用例 CSV
csv_content = '''testcase_name,network_name,op_name,input_shapes,input_dtypes,input_formats,output_shapes,output_dtypes,output_formats,input_ori_shapes,input_ori_formats,output_ori_shapes,output_ori_formats,attributes,input_data_ranges,precision_tolerances,absolute_precision,output_inplace_indexes,output_shape_unknown_indexes,is_enabled,remark,soc_series,priority,dump_file_prefix,manual_input_binaries,manual_golden_binaries
fast_gelu_01,,fast_gelu,"((128, 1024),)","('float32',)","('ND',)","((128, 1024),)","('float32',)","('ND',)","((128, 1024),)","('ND',)","((128, 1024),)","('ND',)",{},"((-1, 1),)",,1e-08,(),(),True,,,0,,(),()
fast_gelu_02,,fast_gelu,"((969, 7188),)","('float16',)","('ND',)","((969, 7188),)","('float16',)","('ND',)","((969, 7188),)","('ND',)","((969, 7188),)","('ND',)",{},"((-2, 2),)",,1e-08,(),(),True,,,0,,(),()
fast_gelu_03,,fast_gelu,"((1024, 1024),)","('bfloat16',)","('ND',)","((1024, 1024),)","('bfloat16',)","('ND',)","((1024, 1024),)","('ND',)","((1024, 1024),)","('ND',)",{},"((-3, 3),)",,1e-08,(),(),True,,,0,,(),()
fast_gelu_04,,fast_gelu,"((8, 2048),)","('float32',)","('ND',)","((8, 2048),)","('float32',)","('ND',)","((8, 2048),)","('ND',)","((8, 2048),)","('ND',)",{},"((-0.5, 0.5),)",,1e-08,(),(),True,,,0,,(),()
fast_gelu_05,,fast_gelu,"((1,),)","('float16',)","('ND',)","((1,),)","('float16',)","('ND',)","((1,),)","('ND',)","((1,),)","('ND',)",{},"((-1, 1),)",,1e-08,(),(),True,,,0,,(),()
fast_gelu_06,,fast_gelu,"((0,),)","('float32',)","('ND',)","((0,),)","('float32',)","('ND',)","((0,),)","('ND',)","((0,),)","('ND',)",{},"((-1, 1),)",,1e-08,(),(),True,,,0,,(),()
'''

with open('TestCases/fast_gelu.csv', 'w') as f:
    f.write(csv_content)

## 2.3 创建 Golden 计算插件

TTK 需要 Golden（CPU基准计算）来对比 NPU 结果精度。对于 FastGelu 算子，PyTorch 没有直接实现，因此需要自定义 Golden 函数。

创建 `ops-test-kit/fast_gelu_golden.py` 文件，包含 Golden 计算逻辑：

In [ ]:
# 创建 Golden 计算插件文件
golden_content = '''# -*- coding: utf-8 -*-
"""Golden calculation for fast_gelu operator"""

import numpy as np
import functools


__golden__ = {
    "kernel": {
        "fast_gelu": "fast_gelu_golden"
    }
}


def fast_gelu_golden(x, **kwargs):
    """
    Golden function for fast_gelu.
    All the parameters (names and order) follow @fast_gelu_def.cpp without outputs.
    All the input Tensors are numpy.ndarray.

    Args:
        **kwargs: {input,output}_{dtypes,ori_shapes,formats,ori_formats},
                  full_soc_version, short_soc_version, testcase_name

    Returns:
        Output tensor
    """
    input_dtype = x.dtype
    if input_dtype.name in ["bfloat16", "float16"]:
        x = x.astype(np.float32)
    
    fuseshape = [1]
    fuseshape[0] = functools.reduce(lambda a, b: a * b, x.shape)
    x = np.reshape(x, fuseshape)
    
    const_1_value = 1
    attr = 1.702
    calc_dtype = x.dtype
    attr_opp = 0 - attr
    attr_half = attr / 2

    const_0 = np.array(attr_opp, calc_dtype)
    const_1 = np.array(const_1_value, calc_dtype)
    denominator = np.multiply(x, const_0)
    denominator = np.exp(denominator)
    denominator = np.add(denominator, const_1)

    denominator = np.reciprocal(denominator)
    result = np.multiply(x, denominator)
    return result.astype(input_dtype, copy=False)
'''

with open('ops-test-kit/fast_gelu_golden.py', 'w') as f:
    f.write(golden_content)

---
# 3. 精度测试

Kernel 模式的完整执行流程：

```
读取CSV → 编译内核（动态/静态） → 生成输入数据 → NPU执行 → 生成Golden（CPU） → 精度比对 → 输出结果
```

## 3.1 基本执行

使用 `kernel` 子命令执行精度测试，通过 `--plugin` 参数指定 Golden 插件：

In [ ]:
# 基本执行（默认动态编译，使用自定义Golden插件）
!cd ops-test-kit && python3 -m ttk kernel -i ../TestCases/fast_gelu.csv --plugin fast_gelu_golden.py

## 3.2 编译模式

TTK 支持 4 种编译模式：

| 参数 | 编译模式 | 说明 |
|------|---------|------|
| `-d`（默认） | 动态编译 | 用动态 shape 编译，经 tiling 得到实际 shape 后执行 |
| `-s` | 静态编译 | 用固定 shape 编译后直接执行 |
| `-c` | 常量编译 | 将指定输入作为常量编译 |
| `-b` | 二进制模式 | 使用预编译的发布内核 |

In [ ]:
# 动态+静态编译同时执行
!cd ops-test-kit && python3 -m ttk kernel -i ../TestCases/fast_gelu.csv --plugin fast_gelu_golden.py -d -s

## 3.3 精度对比方法

支持多种精度对比方法：https://gitcode.com/tan_xin/ops-test-kit/blob/master/docs/%E7%BB%93%E6%9E%9C%E5%88%86%E6%9E%90.md#%E7%B2%BE%E5%BA%A6%E5%AF%B9%E6%AF%94%E6%96%B9%E6%B3%95%E8%AF%B4%E6%98%8E

---
# 4. 性能测试

Kernel 模式默认采集 Profiling 性能数据。

## 4.1 性能相关参数

| 参数 | 说明 | 默认值 |
|------|------|--------|
| `--run` | 执行次数 | 板端 3 次 |
| `--warmup` | Profiling 前预热 | 开启 |
| `--npu-timeout` | NPU 执行超时（ms） | 无限制 |
| `--no-prof` | 禁用 Profiling | 默认开启 |

In [ ]:
# 默认执行（含 Profiling）
!cd ops-test-kit && python3 -m ttk kernel -i ../TestCases/fast_gelu.csv --plugin fast_gelu_golden.py

# 设置执行次数
!cd ops-test-kit && python3 -m ttk kernel -i ../TestCases/fast_gelu.csv --plugin fast_gelu_golden.py --run=5

# 禁用 Profiling
!cd ops-test-kit && python3 -m ttk kernel -i ../TestCases/fast_gelu.csv --plugin fast_gelu_golden.py --no-prof

## 4.2 性能数据文件解析

TTK 执行性能测试后，会在 `ops-test-kit/msprof/op/{testcase_name}/dynamic/` 目录下生成性能数据文件，其中 **`op_summary_*.csv`** 是最重要的算子性能汇总文件。

**文件路径示例：**
```
ops-test-kit/msprof/op/fast_gelu_06/dynamic/PROF_000001_20260526123137549_KCNJAAJDDLBNBFCC/mindstudio_profiler_output/op_summary_20260526123138.csv
```

该文件包含算子的详细执行信息、耗时数据和硬件利用率指标。

In [ ]:
# 查看 op_summary.csv 文件结构
import pandas as pd

# 示例文件路径（请根据实际生成的路径修改）
csv_path = 'ops-test-kit/msprof/op/fast_gelu_06/dynamic/*/mindstudio_profiler_output/op_summary_*.csv'

# 查找实际文件
import glob
files = glob.glob(csv_path)
if files:
    df = pd.read_csv(files[0])

**🔧 AI Core 性能指标（Cube Core）**

| 字段名 | 说明 | 单位 |
|--------|------|------|
| aic_total_cycles | AI Core 总周期数 | cycles |
| aic_mac_time(us) | 矩阵乘累加时间 | 微秒(us) |
| **aic_mac_ratio** | 矩阵乘累加占比（计算密集度） | 比例 |
| aic_scalar_time(us) | 标量计算时间 | 微秒(us) |
| aic_scalar_ratio | 标量计算占比 | 比例 |
| aic_mte1_time(us) | 内存搬运时间（MTE1） | 微秒(us) |
| aic_mte1_ratio | 内存搬运占比（MTE1） | 比例 |
| aic_mte2_time(us) | 内存搬运时间（MTE2） | 微秒(us) |
| aic_mte2_ratio | 内存搬运占比（MTE2） | 比例 |
| aic_fixpipe_time(us) | 数据转换时间 | 微秒(us) |
| aic_fixpipe_ratio | 数据转换占比 | 比例 |
| aic_icache_miss_rate | 指令缓存未命中率 | 比率 |

**⚡ AI Vector Core 性能指标**

| 字段名 | 说明 | 单位 | 示例 |
|--------|------|------|--------|
| aiv_total_cycles | AI Vector Core 总周期数 | cycles | 1287914 |
| **aiv_vec_time(us)** | 向量计算时间 | 微秒(us) | 0.655 |
| **aiv_vec_ratio** | 向量计算占比（计算密集度） | 比例 | 0.666 |
| aiv_scalar_time(us) | 标量计算时间 | 微秒(us) | 0.05 |
| aiv_scalar_ratio | 标量计算占比 | 比例 | 0.053 |
| aiv_mte2_time(us) | 内存搬运时间（MTE2） | 微秒(us) | 0.74 |
| aiv_mte2_ratio | 内存搬运占比（MTE2） | 比例 | 0.74 |
| aiv_mte3_time(us) | 内存搬运时间（MTE3） | 微秒(us) | 0.365 |
| aiv_mte3_ratio | 内存搬运占比（MTE3） | 比例 | 0.365 |
| aiv_icache_miss_rate | 指令缓存未命中率 | 比率 | 0.0 |
| **cube_utilization(%)** | Cube单元利用率 | 百分比 | 0.000 |

**📊 Block Dim 相关字段**

| 字段名 | 说明 |
|--------|------|
| Block Dim | Block维度，并行执行的Block数量 |
| Mix Block Dim | 混合Block维度 |
| HF32 Eligible | HF32格式兼容性标记 |

In [ ]:
# 分析性能数据示例
import pandas as pd
import glob

# 查找最新的性能数据文件
csv_path = 'ops-test-kit/msprof/op/*/dynamic/*/mindstudio_profiler_output/op_summary_*.csv'
files = glob.glob(csv_path)

if files:
    # 读取最新文件
    latest_file = max(files, key=os.path.getmtime)
    df = pd.read_csv(latest_file)

### 4.2.2 性能优化分析指南

**如何解读性能数据：**

1. **找出高耗时算子**：按 `Task Duration(us)` 排序，找出 Top N 耗时算子

2. **分析计算密集度**：
   - **AI Core 算子**：查看 `aic_mac_ratio`（矩阵乘占比）
     - 高占比（>70%）→ 计算密集型，优化计算逻辑
     - 低占比（<30%）→ 内存密集型，优化数据搬运
   - **AI Vector Core 算子**：查看 `aiv_vec_ratio`（向量计算占比）
     - 高占比 → 向量计算为主
     - 低占比 + 高 `aiv_mte2_ratio` → 内存搬运瓶颈

3. **识别内存瓶颈**：
   - 高 `aic_mte1_ratio` / `aic_mte2_ratio`：输入/输出搬运耗时高
   - 高 `aiv_mte2_ratio` / `aiv_mte3_ratio`：Vector Core内存搬运瓶颈
   - **优化方向**：减少搬运次数、优化数据布局

4. **分析等待时间**：`Task Wait Time(us)` 高 → 任务下发延迟，优化调度

5. **多流并行分析**：通过 `Stream ID` 判断任务是否并行执行

**FastGelu 算子特点**（示例数据解读）：
- Task Type: `AI_VECTOR_CORE` → Vector Core 执行
- aiv_vec_ratio: ~66% → 向量计算为主
- aiv_mte2_ratio: ~74% → 内存搬运占主要时间
- Task Duration: ~21-29us → 执行时间短，性能良好

In [ ]:
# 性能优化建议生成
import pandas as pd
import glob

csv_path = 'ops-test-kit/msprof/op/*/dynamic/*/mindstudio_profiler_output/op_summary_*.csv'
files = glob.glob(csv_path)

if files:
    latest_file = max(files, key=os.path.getmtime)
    df = pd.read_csv(latest_file)
    
    # 分析总体耗时
    total_duration = df['Task Duration(us)'].sum()
    avg_duration = df['Task Duration(us)'].mean()
    max_duration = df['Task Duration(us)'].max()
    
    # 找出高耗时算子
    top_slow = df.nlargest(3, 'Task Duration(us)')
    
    # 分析 Vector Core 性能
    vec_tasks = df[df['Task Type'] == 'AI_VECTOR_CORE']
    if not vec_tasks.empty:
        avg_vec_ratio = vec_tasks['aiv_vec_ratio'].mean()
        avg_mte2_ratio = vec_tasks['aiv_mte2_ratio'].mean()

---
# 5. 常用场景示例

以下是典型测试场景的命令示例：

In [ ]:
# 调试单个用例（失败时自动 dump）
!cd ops-test-kit && python3 -m ttk kernel -i ../TestCases/fast_gelu.csv --plugin fast_gelu_golden.py -t fast_gelu_01 --dump-on-fail

In [ ]:
# 仅编译不执行
!cd ops-test-kit && python3 -m ttk kernel -i ../TestCases/fast_gelu.csv --plugin fast_gelu_golden.py --co

# 重跑精度失败的用例
!cd ops-test-kit && python3 -m ttk kernel -i ../TestCases/fast_gelu.csv --plugin fast_gelu_golden.py --rerun=precision_status

# 使用自定义 Golden 插件（已创建）
!cd ops-test-kit && python3 -m ttk kernel -i ../TestCases/fast_gelu.csv --plugin fast_gelu_golden.py

## 5.1 查看设备信息

使用 `ttk info` 查看当前环境和设备信息：

In [ ]:
!cd ops-test-kit && python3 -m ttk info

## 5.2 列出测试用例

使用 `ttk list` 查看 CSV 文件中的测试用例：

In [ ]:
!cd ops-test-kit && python3 -m ttk list -i ../TestCases/fast_gelu.csv

---
# 6. 参考资料

更多详细信息请参考以下官方文档和资源：

**TTK 测试工具文档：**
- [用例生成指南](ops-test-kit/docs/用例生成.md) - CSV格式详细说明
- [Kernel算子测试指南](ops-test-kit/docs/各类算子测试指南/Kernel算子测试指南.md) - Kernel模式完整流程
- [ACLNN算子测试指南](ops-test-kit/docs/各类算子测试指南/ACLNN算子测试指南.md) - ACLNN API测试方法
- [E2E算子测试指南](ops-test-kit/docs/各类算子测试指南/E2E算子测试指南.md) - E2E框架测试方法
- [结果分析](ops-test-kit/docs/结果分析.md) - 测试结果解读与分析
- [FAQ一本通](ops-test-kit/docs/FAQ和问题自定位指南/FAQ一本通.md) - 常见问题解答

**FastGelu 算子相关：**
- [FastGelu算子说明](../ops-nn/activation/fast_gelu/README.md) - 算子功能、参数、约束说明
- [FastGelu API文档](../ops-nn/activation/fast_gelu/docs/aclnnFastGelu.md) - ACLNN接口调用方法
- [FastGelu示例代码](../ops-nn/activation/fast_gelu/examples/) - ACLNN调用示例

**CANN 开发资源：**
- [Ascend C 开发指南](https://www.hiascend.com/document) - 官方开发文档
- [算子开发指南](https://www.hiascend.com/document) - 算子开发完整流程
- [性能优化指南](https://www.hiascend.com/document) - 性能调优最佳实践

In [ ]:
# 快速访问链接:
# TTK 官方文档路径:
#   用例生成: ops-test-kit/docs/用例生成.md
#   Kernel测试: ops-test-kit/docs/各类算子测试指南/Kernel算子测试指南.md
#   ACLNN测试: ops-test-kit/docs/各类算子测试指南/ACLNN算子测试指南.md
#   E2E测试: ops-test-kit/docs/各类算子测试指南/E2E算子测试指南.md
#   结果分析: ops-test-kit/docs/结果分析.md
#   FAQ指南: ops-test-kit/docs/FAQ和问题自定位指南/FAQ一本通.md
# FastGelu 算子资源:
#   算子说明: ops-nn/activation/fast_gelu/README.md
#   API文档: ops-nn/activation/fast_gelu/docs/aclnnFastGelu.md
#   示例代码: ops-nn/activation/fast_gelu/examples/

---

# 课后练习

完成以下题目，检验你对 Kernel 测试的理解：

1. （判断题）TTK 使用 CSV 文件定义测试用例，支持批量测试多个场景。    

2. （判断题）对于 PyTorch 有内置实现的算子，TTK 无需创建自定义 Golden 插件。    

3. （单选题）TTK 的编译模式中，动态编译（-d）模式的特点是什么？  
    A. 用固定 shape 编译后直接执行  
    B. 用动态 shape 编译，经 tiling 得到实际 shape 后执行  
    C. 将指定输入作为常量编译  
    D. 使用预编译的发布内核  

4. （单选题）性能数据文件 op_summary.csv 中，哪个字段表示向量计算占比？  
    A. aiv_total_cycles  
    B. aiv_vec_ratio  
    C. Task Duration(us)  
    D. aiv_mte2_ratio  

5. （单选题）精度对比时，判定条件是什么？  
    A. |npu - golden| < atol  
    B. |npu - golden| < rtol  
    C. |npu - golden| <= atol + rtol × |golden|  
    D. |npu - golden| == 0  

**执行以下代码获取答案。**

---

上一节：[3.1 SIMT 算子开发](./03.01_simt_operator_development.ipynb)

In [ ]:
!cat ./answer/03.05_answer.txt
